In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from nltk.stem import PorterStemmer
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

import tensorflow as tf




In [2]:
fake=pd.read_csv('/kaggle/input/fake-news-detection/fake.csv')
true=pd.read_csv('/kaggle/input/fake-news-detection/true.csv')

In [3]:
fake

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"
...,...,...,...,...
23476,McPain: John McCain Furious That Iran Treated ...,21st Century Wire says As 21WIRE reported earl...,Middle-east,"January 16, 2016"
23477,JUSTICE? Yahoo Settles E-mail Privacy Class-ac...,21st Century Wire says It s a familiar theme. ...,Middle-east,"January 16, 2016"
23478,Sunnistan: US and Allied ‘Safe Zone’ Plan to T...,Patrick Henningsen 21st Century WireRemember ...,Middle-east,"January 15, 2016"
23479,How to Blow $700 Million: Al Jazeera America F...,21st Century Wire says Al Jazeera America will...,Middle-east,"January 14, 2016"


In [4]:
pd.Series(0,index=true.index,name='label')

0        0
1        0
2        0
3        0
4        0
        ..
21412    0
21413    0
21414    0
21415    0
21416    0
Name: label, Length: 21417, dtype: int64

In [5]:
true_df=pd.concat([true['title'],pd.Series(0,index=true.index,name='label')],axis=1)
fake_df=pd.concat([true['title'],pd.Series(1,index=true.index,name='label')],axis=1)

In [6]:
news_df=pd.concat([true_df,fake_df],axis=0).sample(frac=1.0,random_state=34).reset_index(drop=True)

In [7]:
news_df

,title,label
0,U.S. lawmakers reach deal for Senate vote on R...,1
1,Spanish police seize ballot boxes in Catalan r...,1
2,Second House panel approves Obamacare replacem...,1
3,"Macron urges the French to value success, reje...",1
4,Exclusive: European envoys take fight for Iran...,0
...,...,...
42829,White House demands Cuba take 'concrete' refor...,0
42830,Goldman's Cohn eyed for top Trump budget post:...,1
42831,U.S. regulator asks Trump not to dismantle cla...,1
42832,Iran sentences 'Mossad agent' to death over sc...,1


# Preprocessing

In [8]:
ps=PorterStemmer()
def process_title(title):
    new_title=title.lower()
    new_title=re.sub(r'\$[^\s]+','dollar',new_title)
    new_title=re.sub(r'[^a-z0-9\s]','',new_title)
    new_title=re.sub(r'[0-9]+','number',new_title)
    new_title=new_title.split(' ')
    new_title= list(map(lambda x :ps.stem(x),new_title))
    new_title= list(map(lambda x : x.strip(),new_title))
    if ''in new_title:
        new_title.remove('')
                    
                    
    return new_title

In [9]:
titles=news_df['title'].apply(process_title)

In [10]:
labels=np.array(news_df['label'])

In [11]:
labels

array([1, 1, 1, ..., 1, 1, 0])

In [12]:
titles.shape

(42834,)

In [13]:
# Get size  of vocabulary
vocabulary=set()

for title in titles:
    for word in title:
        if word not in vocabulary:
            vocabulary.add(word)


            
            
vocab_length=len(vocabulary)            
# Get max length of sequence

max_seq_length=0

for titel in titles:
    if len(title)>max_seq_length :
        max_seq_length=len(title)

        # Print Result  
print('vocab_length',vocab_length)
print('max seuence length',max_seq_length)

vocab_length 9862
max seuence length 10


In [14]:
tokenizer=Tokenizer(num_words=vocab_length)
tokenizer.fit_on_texts(titles)


sequences=tokenizer.texts_to_sequences(titles)
word_index=tokenizer.word_index
model_inputs=pad_sequences(sequences,maxlen=max_seq_length)

In [15]:
sequences[0]

[4, 72, 490, 38, 8, 12, 30, 5, 18, 83]

In [16]:
titles[0]

['us',
 'lawmak',
 'reach',
 'deal',
 'for',
 'senat',
 'vote',
 'on',
 'russia',
 'sanction']

In [17]:
for i in ['us',
             'lawmak',
             'reach',
             'deal',
             'for',
             'senat',
             'vote',
             'on',
             'russia',
             'sanction']:
    print(word_index[i])

4
72
490
38
8
12
30
5
18
83


In [18]:
pd.DataFrame(model_inputs)

,0,1,2,3,4,5,6,7,8,9
0,4,72,490,38,8,12,30,5,18,83
1,0,0,614,66,880,1007,2712,3,177,395
2,0,0,0,701,11,124,218,170,538,44
3,47,325,1,1773,1033,244,34,7,2015,3556
4,277,118,104,8,53,91,38,1,4,78
...,...,...,...,...,...,...,...,...,...,...
42829,11,517,306,118,3011,122,315,243,1061,838
42830,1917,1523,349,8,85,2,126,358,503,48
42831,0,4,447,189,2,24,1,1919,4404,55
42832,0,53,439,5928,1073,1,292,15,2061,56


In [19]:
X_train,X_test,y_train,y_test=train_test_split(model_inputs,labels,train_size=0.7)

In [20]:
embedding_dim=64
inputs=tf.keras.Input(shape=(max_seq_length,))
embedding=tf.keras.layers.Embedding(
                                    input_dim=vocab_length,
                                    output_dim=embedding_dim,
                                    input_length=max_seq_length )(inputs)
gru=tf.keras.layers.GRU(units=embedding_dim)(embedding)
outputs=tf.keras.layers.Dense(1,activation='sigmoid')(gru)

model=tf.keras.Model(inputs=inputs,outputs=outputs)


model.compile(
                optimizer='adam',
                loss='binary_crossentropy',
                 metrics=['accuracy',
                            tf.keras.metrics.AUC(name='auc')])


batch_size=32
epochs=2



history=model.fit(
                    X_train,
                    y_train,
                    validation_split=0.2,
                    batch_size=batch_size,
                    epochs=epochs,
                    callbacks=[
                        tf.keras.callbacks.ReduceLROnPlateau(),
                        tf.keras.callbacks.ModelCheckpoint('model.h5',save_best_only=True)
                    ]
)



Epoch 1/2
750/750 [==============================] - 17s 18ms/step - loss: 0.6936 - accuracy: 0.4980 - auc: 0.4982 - val_loss: 0.6940 - val_accuracy: 0.4907 - val_auc: 0.4954
Epoch 2/2
750/750 [==============================] - 13s 18ms/step - loss: 0.6935 - accuracy: 0.5013 - auc: 0.4977 - val_loss: 0.6936 - val_accuracy: 0.4952 - val_auc: 0.5114


# Result

In [21]:
fig=px.line(
        history.history,
            y=['loss','val_loss'],
            labels={"x":"Epochs","y":"Loss"},
            title='Loss Over Time')
fig.show()

In [22]:
np.argmin( history.history['val_loss'])

1

In [23]:
fig=px.line(
        history.history,
            y=['auc','val_auc'],
            labels={"x":"Epochs","y":"AUC"},
            title='AUC Over Time')
fig.show()

In [24]:
model.load_weights('./model.h5')

In [25]:
model.evaluate(X_test,y_test)

402/402 [==============================] - 2s 5ms/step - loss: 0.6936 - accuracy: 0.4979 - auc: 0.5045


[0.6935639977455139, 0.4978600740432739, 0.5044755935668945]